# Spatial | Spatial Occupancy
-----------------------------------------

## Setup and Configuration

In [ ]:
##### Imports ##########################################################################

#==== Generic Imports ==================================================================
from pathlib import Path
from collections.abc import Callable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

#==== Movement Imports =================================================================
import movement.kinematics
import movement.utils.vector

#==== Data-Conduit Imports =============================================================
from data_conduit.datastructures import slice_stream, slice_stream_per_trial
from data_conduit.integrations.DLC.pose import pose_to_movement
from data_conduit.refactor_qc import (
    TRAINING_FIRST_MID_LAST_DETAIL,
    training_spec,
    filter_trials,
)

#==== Figure-Project Imports ===========================================================
from movement_figures.data_template.loading import build_datastructure
from movement_figures.misc import visual_streams_check, configure_simplified_trial_df
from movement_figures.timeseries_template.annotations import (
    AnnotationResult,
    annotate_qc_timeseries,
)

import movement.plots
from movement_figures.video import read_session_video_frame


### Notebook Settings ######################################################

pd.set_option("display.max_columns", None)  # Changes default pandas display settings to show all columns in a dataframe.
pd.set_option("display.max_rows", None)  # Changes default pandas display settings to show all rows in a dataframe.

In [ ]:
##### Setup Parameters #################################################################

# === 1| Select Mice, Days and Recording Folders Before Reading Files =============
ROOT = Path("/media/sepi/Elements/PathIntegrationProtocol/BonsaiOutput/Training")

LEVEL_SELECTORS = {
    'l0_selector': 'MR_M01569515', 
    'l1_selector': 'Day21'
    }                                                                                       # Example choices belong here: l0_selector and l1_selector.
# INCLUDE_SESSIONS = ('2026-06-21T093349Z',)                                                # Optional list of recording folder names; None keeps selected folders.
# STREAMS = ('trials', 'events', 'video', 'dlc')                                            # These figures require events/trials and aligned pose; other sources remain available.


#=== 2| Set the trial filtering parameters for the experiment protocol ============
spec = training_spec(TRAINING_FIRST_MID_LAST_DETAIL)                                        # Flattens `{training_detail}` table into per-session spec. 

# included_sessions = training_session_names(spec)                                          # Returns a list of session folder names that match the spec.
#   Not used here, as sessions have been filtered in another way. If other sessions 
#   not in the above spec are used, then applying the `filter_trials` function with 
#   the above spec could result in incorrect filtering.


In [ ]:
##### Load DataStructure ###############################################################

#=== Load data ========================================================================

fig_ds_obj = build_datastructure(
    root               = ROOT,                                                                 # Directory above the named hierarchy levels.
    level_names        = ("mouseID", "day"),                                                   # Lab layout: root / mouseID / day / session.
    streams            = ("trials", "session_settings", "video", "dlc"),                       # Data streams to be included in the datastructure.
    include            = None,                                                                 # Keep these session folder names; None keeps all.
    exclude            = None,                                                                 # Alternatively omit these session folder names.
    raw_events         = True,                                                                 # Convenience alias for adding "events" to the requested sources.
    device_yaml        = "./device.yml",                                                       # Lab Nosepoke board register definitions.
    soundcard_yaml     = "./soundcard.yml",                                                    # Lab SoundCard register definitions.
    nosepoke_count     = 18,                                                                   # Port count used by the existing Q_C parser.
    trial_start_buffer = 0.0,                                                                  # Preserve the existing lab parser setting.
    dlc_file_format     = "auto",                                                              # Existing reader prefers CSV, otherwise HDF5.
    trial_spec = None,                                                                         # Optional experiment rules; None keeps the existing Q_C specification. 
    #    Not used here, as sessions are already selected. 
    #    If session is not one in the spec, then subsequent use of 
    #   `filter_trials` to apply correct experiment protocol rules could be incorrect.
    
    **LEVEL_SELECTORS,
)


#=== Select and load the streams of interest from datastructure. ======================
fig_ds_obj.select()
streams = fig_ds_obj.load()                                                                    # Load is required to access the selected streams.

#=== Create convenient compact trial dataframe without some unnecessary columns. ======
streams['compact_trials'] = configure_simplified_trial_df(              
    trial_df = streams['trials'],
    modify_in_place = False,                                                                   # Whether to modify the trial dataframe in place or return a new dataframe. 
)
'''More useful for visual inspection and analysis. 
Check the misc.py file for the default columns to keep and drop.'''


#=== Apply Trial Filtering Protocol to construct new stream objects ===================

# Full filtered trials
streams['filtered_trials'] = filter_trials(
    trials = streams['trials'],
    spec = spec,
    # session_column = 'session', mouse_column = 'mouseID', trial_index_column = 'trial_index', led_column = 'LED', report = True,
    #                                                            # Leave params untouched for now, let the function use its defaults.
)

# Filtered compact trials 
streams['filtered_compact_trials'] = filter_trials(
    trials = streams['compact_trials'],
    spec = spec,
    # session_column = 'session', mouse_column = 'mouseID', trial_index_column = 'trial_index', led_column = 'LED', report = True,
    #                                                            # Leave params untouched for now, let the function use its defaults.
)

#=== Merge Pose & Confidence Streams ==================================================
streams['dlc:movement'] = pose_to_movement(
    slice_stream(streams["dlc:position"],   
                #selectors={"session": session_id}
    ),
    slice_stream(streams["dlc:confidence"], 
                 #selectors={"session": session_id}
    ),
)

#=== Visual streams check =============================================================
visual_streams_check(
    loaded_ds_object = streams,                                                                # Loaded dataset object containing all streams of interest.
    show_stream_shapes = True,                                                                 # Whether to display the shapes of the streams.
    show_stream_dtypes = True,                                                                 # Whether to display the data types of the streams.
    display_streams = False                                                                    # Whether to display the actual stream data as a table.
)



## Select Trials to Show

In [ ]:
##### Trial Selection ##################################################################

#==== Extract Trial Range ============
selected_trials = slice_stream(streams['filtered_trials'][:8])              # Select the first n trials

#==== Extract Recording Identifier ===
sessionID = selected_trials['session'].unique().item()                      # Extract unique sID from selected trials, remove duplicates, ensure single value.

#==== Select Tracked Positions =======
recording = slice_stream(stream = streams['dlc:movement'],                  # selectors{} here keeps movement data belonging to the specified session only.
                        selectors = {'session' : sessionID }, 
)

position = recording['position']                                            # ['position'] extracts positional data from the movement stream.

position = position.sel(individual='individual_0',                          # .sel() selects the specified individual from the positional data.
                        drop=True,                                          # drop=True removes the individual dimension from the resulting DataArray, but leaves other dims intact.
)


In [ ]:
##### Read the Matching Video Background ################################################

background_frame, video_path = read_session_video_frame(
    fig_ds_obj.sessions[sessionID].path,                     # Same recording and undistorted pixel coordinates as position.
)

## Plot One Group and Phase

Select the outcome group and phase first, then pass those position samples to 
`plot_ax_spatial_occupancy`. The function draws one panel on the supplied axis.


In [ ]:
##### Plot Spatial Occupancy on Ax ######################################################


def plot_ax_spatial_occupancy(
    ax: plt.Axes,
    points: xr.DataArray,
    background_frame: np.ndarray,
    bins: int = 40,
    vmax: float | None = 100.0,
    subtitle: str | None = "Spatial Occupancy",
    alpha: float = 0.40,
    **kwargs,
) -> tuple[plt.Axes, dict]:
    """Plot one selected group and phase over its matching video frame.

    Parameters
    ----------
    ax : plt.Axes
        Axis on which to draw one panel.
    points : xr.DataArray
        Already-selected body positions with dimensions (time, space), in pixels.
        Select the recording, individual, trial outcomes and phases beforehand.
    background_frame : np.ndarray
        Matching unscaled RGB video frame. It sets the image limits and bin range.
    bins : int
        Number of equal-width spatial bins along each image axis.
    vmax : float or None
        Upper colour limit in samples per bin. Use the same value for comparison panels.
        None lets this panel choose its own upper colour limit.
    subtitle : str or None
        Panel title; None omits it.
    alpha : float
        Histogram opacity over the video background.
    **kwargs
        Passed to movement.plots.plot_occupancy and its underlying hist2d call.

    Returns
    -------
    plt.Axes
        Supplied axis containing the overlay and movement's colour bar.
    dict
        Histogram counts h and their xedges/yedges, as returned by movement.
    """
    #==== 0| Validation Checks ==========================================================
    if ax is None:
        raise ValueError("ax must be provided")
    if points is None:
        raise ValueError("points must be provided")

    #==== 1| Use the Same Pixel Extent for the Video and the Histogram ====================
    height, width = background_frame.shape[:2]
    extent = (-0.5, width - 0.5, height - 0.5, -0.5)
    bin_range = ((-0.5, width - 0.5), (-0.5, height - 0.5))
    ax.imshow(background_frame, origin="upper", extent=extent)

    #==== 2| Let movement Count the Supplied Position Samples =============================
    _, _, occupancy = movement.plots.plot_occupancy(
        da=points, ax=ax, bins=bins, range=bin_range,
        vmin=0, vmax=vmax, alpha=alpha, **kwargs,
    )

    #==== 3| Keep the Video's Downward-Positive y Coordinates =============================
    ax.set(xlabel="x (Pixels)", ylabel="y (Pixels)", aspect="equal",
           xlim=extent[:2], ylim=extent[2:])
    if subtitle is not None:
        ax.set_title(subtitle)
    return ax, occupancy

In [ ]:
#==== Choose Which Outcomes to Compare Before Plotting ==================================
OUTCOME_GROUPS = {"Success": ("Success",)}                   # Default: success trials only.
# OUTCOME_GROUPS = {"Success": ("Success",), "Failure": ("Failure",), "Miss": ("Miss",)}
# OUTCOME_GROUPS = {"Success": ("Success",), "Failure + Miss": ("Failure", "Miss")}

PHASES = {"Selected trials": "trial", "Outbound": "outbound", "Inbound": "inbound"}
BINS = 40
VMAX = 100.0 

##### Slice the Actual Samples for Each Occupancy Panel #################################

body_position = position.sel(keypoint="body", drop=True)
occupancy_points = {}
panel_trials = {}

#==== Select Outcome Rows First, Then Slice the Matching Body Positions =================
for outcome_label, outcomes in OUTCOME_GROUPS.items():
    outcome_trials = selected_trials.loc[selected_trials["outcome"].isin(outcomes)]
    if outcome_trials.empty:
        print(f"No selected trials for {outcome_label}; no panel will be drawn.")
        continue

    for phase_label, segment in PHASES.items():
        paths = slice_stream_per_trial(
            stream=body_position,
            trials=outcome_trials,
            segment=segment,
            time_coord="time",
        )
        label = f"{outcome_label} | {phase_label}"
        occupancy_points[label] = xr.concat(paths, dim="time")
        panel_trials[label] = len(outcome_trials)

display(pd.Series(panel_trials, name="Number of selected trials"))




In [ ]:
##### Select One Outcome Group and Phase ###############################################

OUTCOMES = ("Success",)                                     # One outcome group; e.g. ("Failure", "Miss").
SEGMENT = "trial"                                          # One phase: "trial", "outbound" or "inbound".
BINS = 40
VMAX = 100.0

body_position = position.sel(keypoint="body", drop=True)
outcome_trials = selected_trials.loc[selected_trials["outcome"].isin(OUTCOMES)]

if outcome_trials.empty:
    print(f"No selected trials for {OUTCOMES}; no panel will be drawn.")
else:
    paths = slice_stream_per_trial(
        stream     = body_position,
        trials     = outcome_trials,
        segment    = SEGMENT,
        time_coord = "time",
    )
    points = xr.concat(paths, dim="time")

    #==== Draw One Occupancy Panel ======================================================
    fig, ax = plt.subplots(figsize=(6, 5))

    ax, occupancy = plot_ax_spatial_occupancy(
        ax               = ax,
        points           = points,
        background_frame = background_frame,
        bins             = BINS,
        vmax             = VMAX,
        subtitle         = f"{' + '.join(OUTCOMES)} | {SEGMENT}",
        alpha            = 0.40,
    )

    plt.show()


## Compare Outcome Groups and Phases

The expanded function filters the selected trial table and slices the matching position samples. It calls `plot_ax_spatial_occupancy` for each panel, arranging outcome groups as rows and phases as columns. All panels use the same background, bin range and supplied colour limits.


In [ ]:
##### Plot Outcome Groups and Phases on Ax #############################################


def plot_ax_spatial_occupancy_groups(
    ax: plt.Axes,
    points: xr.DataArray,
    selected_trials: pd.DataFrame,
    background_frame: np.ndarray,
    outcome_groups: dict[str, tuple[str, ...]] | None = None,
    phases: dict[str, str] | None = None,
    bins: int = 40,
    vmax: float | None = 100.0,
    subtitle: str | None = "Spatial Occupancy",
    alpha: float = 0.40,
    **kwargs,
) -> tuple[plt.Axes, dict, dict]:
    """Compare outcome groups and phases using the single-panel occupancy function.

    Parameters
    ----------
    ax : plt.Axes
        Empty axis reserved for the comparison; the caller sets the figure size.
        One panel uses this axis directly. Multiple panels use ax.child_axes.
    points : xr.DataArray
        Body positions with dimensions (time, space), in pixels, for one recording
        and individual. Keep the session coordinate required by trial slicing.
        Supply the samples covering all requested trials and phases.
    selected_trials : pd.DataFrame
        Trial rows from that recording, including outcome, session and the time
        bounds for each requested phase. Outcome filtering happens within this table.
    background_frame : np.ndarray
        Matching unscaled RGB video frame, passed to the single-panel function.
    outcome_groups : dict[str, tuple[str, ...]] or None
        Row labels mapped to outcomes, in display order. None uses
        {"Success": ("Success",)}. For a combined group use, for example,
        {"Failure + Miss": ("Failure", "Miss")}. Groups with no trials are skipped;
        if no groups remain, a ValueError is raised before drawing.
    phases : dict[str, str] or None
        Column labels mapped to trial segment names, in display order. None uses
        {"Selected trials": "trial", "Outbound": "outbound", "Inbound": "inbound"}.
        A phase with no samples keeps its panel and has zero histogram counts.
    bins : int
        Number of equal-width spatial bins along each image axis in every panel.
    vmax : float or None
        Common upper colour limit in samples per bin. None lets each panel choose
        its own upper colour limit. Histogram counts are not normalised.
    subtitle : str or None
        Title above the comparison. None omits it; group and phase labels remain.
    alpha : float
        Histogram opacity over the video background.
    **kwargs
        Passed to plot_ax_spatial_occupancy for every panel.

    Returns
    -------
    plt.Axes
        Original axis; multiple panels are available through ax.child_axes.
    dict
        Histogram results keyed by "outcome label | phase label". Each result has
        h, xedges and yedges, as returned by the single-panel function.
    dict
        Number of selected trial rows per panel, under the same labels. These
        counts include trials whose requested phase contains no position samples.
    """
    #==== 0| Validation Checks ==========================================================
    if ax is None:
        raise ValueError("ax must be provided")
    if points is None:
        raise ValueError("points must be provided")
    if selected_trials is None:
        raise ValueError("selected_trials must be provided")

    if outcome_groups is None:
        outcome_groups = {"Success": ("Success",)}
    if phases is None:
        phases = {"Selected trials": "trial", "Outbound": "outbound", "Inbound": "inbound"}
    if not outcome_groups:
        raise ValueError("outcome_groups must contain at least one group")
    if not phases:
        raise ValueError("phases must contain at least one phase")

    #==== 1| Select Outcome Rows, Then Slice the Matching Position Samples ===============
    occupancy_points = {}
    panel_trials = {}
    group_labels = []

    for outcome_label, outcomes in outcome_groups.items():
        outcome_trials = selected_trials.loc[selected_trials["outcome"].isin(outcomes)]
        if outcome_trials.empty:
            print(f"No selected trials for {outcome_label}; no panel will be drawn.")
            continue

        group_labels.append(outcome_label)
        for phase_label, segment in phases.items():
            paths = slice_stream_per_trial(
                stream     = points,
                trials     = outcome_trials,
                segment    = segment,
                time_coord = "time",
            )
            label = f"{outcome_label} | {phase_label}"
            occupancy_points[label] = xr.concat(paths, dim="time")
            panel_trials[label] = len(outcome_trials)

    if not group_labels:
        raise ValueError("No selected trials match the requested outcome groups.")

    #==== 2| Calculate Rows and Columns for the Supplied Axis ============================
    row_count = len(group_labels)
    column_count = len(phases)
    single_panel = row_count == 1 and column_count == 1

    row_height = 1 / (row_count + 0.50 * (row_count - 1))
    column_width = 1 / (column_count + 0.50 * (column_count - 1))
    row_gap = 0.50 * row_height
    column_gap = 0.50 * column_width

    if not single_panel:
        ax.set_axis_off()                                    # Use the supplied axis as the container for the panels.

    #==== 3| Draw Every Panel Using the Single-Group Function ============================
    occupancy = {}
    for row, outcome_label in enumerate(group_labels):
        for column, phase_label in enumerate(phases):
            label = f"{outcome_label} | {phase_label}"
            if single_panel:
                panel_ax = ax
            else:
                left = column * (column_width + column_gap)
                bottom = 1 - (row + 1) * row_height - row * row_gap
                panel_ax = ax.inset_axes([left, bottom, column_width, row_height])

            panel_title = label
            if single_panel and subtitle is not None:
                panel_title = f"{subtitle}\n{label}"

            _, occupancy[label] = plot_ax_spatial_occupancy(
                ax               = panel_ax,
                points           = occupancy_points[label],
                background_frame = background_frame,
                bins             = bins,
                vmax             = vmax,
                subtitle         = panel_title,
                alpha            = alpha,
                **kwargs,
            )

            if not single_panel:
                # Keep the colour bar beside its inset panel as the figure resizes.
                histogram = panel_ax.collections[-1]
                histogram.colorbar.remove()
                colorbar_ax = panel_ax.inset_axes([1.05, 0, 0.05, 1])
                colorbar = panel_ax.figure.colorbar(histogram, cax=colorbar_ax)
                if colorbar.solids is not None:
                    colorbar.solids.set(alpha=1.0)

    #==== 4| Add the Overall Title and Return the Panel Results ==========================
    if not single_panel and subtitle is not None:
        ax.set_title(subtitle, pad=36)
    return ax, occupancy, panel_trials


In [ ]:
#==== Choose Which Outcomes to Compare Before Plotting ==================================
# OUTCOME_GROUPS = {"Success": ("Success",)}                   # Default: success trials only.
# OUTCOME_GROUPS = {"Success": ("Success",), "Failure": ("Failure",), "Miss": ("Miss",)}
OUTCOME_GROUPS = {"Success": ("Success",), "Failure + Miss": ("Failure", "Miss")}

PHASES = {"Selected trials": "trial", "Outbound": "outbound", "Inbound": "inbound"}
BINS = 40
VMAX = 100.0

#==== Draw Outcome Groups as Rows and Phases as Columns =================================
body_position = position.sel(keypoint="body", drop=True)
fig, ax = plt.subplots(figsize=(5 * len(PHASES), 4 * len(OUTCOME_GROUPS)))

ax, occupancy_by_panel, panel_trials = plot_ax_spatial_occupancy_groups(
    ax               = ax,
    points           = body_position,
    selected_trials  = selected_trials,
    background_frame = background_frame,
    outcome_groups   = OUTCOME_GROUPS,
    phases           = PHASES,
    bins             = BINS,
    vmax             = VMAX,
    subtitle         = "Spatial Occupancy",
    alpha            = 0.40,
)

display(pd.Series(panel_trials, name="Number of selected trials"))
plt.show()
